# Evaluación de Programación - Análisis Académico
Este notebook contiene la implementación de clases para gestionar información académica de personas, alumnos y profesores, junto con el procesamiento y análisis de datos cargados desde un archivo CSV. El objetivo es demostrar la capacidad de estructurar código Python, realizar cálculos y presentar resultados de manera clara.

## Reglas de la Evaluación
A continuación, se adjuntan las reglas de la evaluación tal como fueron proporcionadas, para referencia y cumplimiento.

1. Este trabajo debe resolverse de manera individual. Se prohíbe compartir código, ya sea total o parcialmente.
2. La fecha de entrega es el domingo 28 de junio de 2026 a las 23:59 mediante Canvas. No se aceptan entregas por correo ni fuera de plazo.
3. La entrega consiste en un único archivo .ipynb con todas las partes implementadas en orden. El archivo debe ejecutarse sin errores de principio a fin con un kernel limpio (Restart & Run All).
4. Se permite utilizar herramientas de IA generativa únicamente como referencia (consulta de sintaxis, comprensión de conceptos). Está prohibido usarla para generar, corregir o completar el código entregado. Si se usó IA generativa en alguna parte, debes incluir un archivo anexo (.txt o .pdf) con: el prompt exacto utilizado, la respuesta entregada por la herramienta, y una breve justificación de cómo la usaste. La omisión de este anexo cuando corresponde se considerará uso indebido.
5. Soluciones con similitud sospechosa pueden derivar en nota mínima para ambas partes, sin distinción de quién copió a quién.
6. Ante sospecha de trabajo indebido, se llamará al estudiante a evaluación oral donde se pedirá que explique cualquier parte de su código. La incapacidad de explicar el propio código es causal de anulación de la entrega y nota mínima, además de ser derivado al comité de ética.

## 1. Definición de Clases
Se definen tres clases principales: `Personas`, `Alumno` y `Profesor`. La clase `Personas` es la clase base que contiene atributos comunes como RUT, nombre, apellido y correo, además de un método para validar el RUT. Las clases `Alumno` y `Profesor` heredan de `Personas` y añaden atributos y métodos específicos para sus roles.

In [ ]:
class Personas():
    def __init__(self, rut, nombre, apellido, correo):
        self.rut = rut
        self.nombre = nombre
        self.apellido = apellido
        self.correo = correo

    def validarut(self):
        # Eliminar guiones y puntos del RUT
        sacar = self.rut.replace("-", "").replace(".", "")
        # Obtener el dígito verificador
        verificador = sacar[-1]
        # Invertir el RUT sin dígito verificador
        voltear = sacar[len(sacar)-2::-1]
        
        suma = 0
        factor = 2
        for validar in voltear:
            suma += int(validar) * factor
            factor += 1
            if factor > 7:
                factor = 2
           
        resto = 11 - (suma % 11) # Calcular el dígito verificador esperado

        if resto == 11:
            resto = 0
        elif resto == 10:
            resto = "K"
       
        if str(resto).upper() == verificador.upper():
            return True
        return False

    def __repr__(self):
        return f"RUT: {self.rut} | Nombre: {self.nombre} {self.apellido} | Correo: {self.correo}"
       
class Alumno(Personas):
    def __init__(self, rut, nombre, apellido, carrera, asignatura, nota1, nota2, nota3, asistencia, correo):
        super().__init__(rut, nombre, apellido, correo)
        self.carrera = carrera
        self.asignatura = asignatura
        self.nota1 = float(nota1)
        self.nota2 = float(nota2)
        self.nota3 = float(nota3)
        self.asistencia = int(asistencia)

    def promedio(self):
        notas_totales = (self.nota1, self.nota2, self.nota3)
        prom = round(sum(notas_totales)/len(notas_totales),1)
        return prom

    def situacion_academica(self):
        if self.promedio() >= 4.0 and self.asistencia >= 75:
            return "Aprobado"
        return "Reprobado"
    
    def necesita_apoyo(self):
        if self.asistencia < 80 or self.promedio() < 4.5: return True
        return False
   
    def __repr__(self):
        return f"Alumno: {self.nombre} {self.apellido} | Asignatura: {self.asignatura} | Promedio: {self.promedio()} | Asistencia: {self.asistencia}% | Correo: {self.correo}"
    

class Profesor(Personas):
    def __init__(self, rut, nombre, apellido, departamento, asignatura, horas_contrato, correo):
        super().__init__(rut, nombre, apellido, correo)
        self.departamento = departamento
        self.asignatura = asignatura
        self.horas_contrato = horas_contrato

    def tipo_contrato(self):
       if self.horas_contrato >= 20:
           return "Jornada Completa"
       return "Jornada Parcial"
    def __repr__(self):
        return f"Profesor: {self.nombre} {self.apellido} | Asignatura: {self.asignatura} | Horas: {self.horas_contrato} | Tipo Contrato: {self.tipo_contrato()}"


## 2. Carga y Procesamiento de Datos
Los datos de personas académicas (alumnos y profesores) se cargan desde un archivo CSV (`personas_academicas.csv`). Se itera sobre cada fila para instanciar objetos `Alumno` o `Profesor` y almacenarlos en listas separadas, así como en una lista general de `personas`. Se incluye manejo de errores para la ausencia del archivo.

**Nota:** Para que este código funcione correctamente, debe existir un archivo `personas_academicas.csv` en el mismo directorio que este notebook, con el formato esperado (encabezado en la primera línea, datos separados por comas, y las columnas en el orden que se utilizan para instanciar las clases `Alumno` y `Profesor`).

In [ ]:
# Inicialización de listas y contadores
profesores = []
alumnos = []
personas = []
rut_validos = 0
rut_invalidos = 0
alumnos_aprobados = 0
alumnos_reprobados = 0
mejor_promedio = 0
mejor_alumno = None
total_horas_contratadas = 0
profesor_mas_horas = []
max_horas = 0
profesores_por_asignatura = {}
alumnos_por_asignatura = {}

try:
    with open ("personas_academicas.csv", "r", encoding="utf-8") as pa:
        leer = pa.readlines()
        for buscar in leer[1::]: # Ignorar la primera línea (encabezados)
            mostrar = buscar.strip().split(",")
            
            tipo = mostrar[0].upper()
            if tipo ==  "ALUMNO":
                estudiante = Alumno(
                rut=mostrar[1],
                nombre=mostrar[2],
                apellido=mostrar[3],
                carrera=mostrar[4],
                asignatura=mostrar[5],
                nota1=mostrar[6],
                nota2=mostrar[7],
                nota3=mostrar[8],
                asistencia=mostrar[9],
                correo=mostrar[11]
                )
                alumnos.append(estudiante)
                personas.append(estudiante)

            elif tipo == "PROFESOR":
                profesor = Profesor(
                rut=mostrar[1],
                nombre=mostrar[2],
                apellido=mostrar[3],
                departamento=mostrar[4],
                asignatura=mostrar[5],
                horas_contrato=int(mostrar[10]),
                correo=mostrar[11]
                )
                profesores.append(profesor)
                personas.append(profesor)
except FileNotFoundError:
    print("Error: El archivo \"personas_academicas.csv\" no se encontró. Asegúrate de que esté en el mismo directorio que este notebook.")
except Exception as e:
    print(f"Ocurrió un error al cargar los datos: {e}")


## 3. Análisis de Datos y Presentación de Resultados
Se realizan diversas operaciones para analizar los datos cargados y se presentan los resultados de manera estructurada.

### 3.1 Validación de RUTs
Se valida el RUT de cada persona y se cuenta cuántos son válidos e inválidos.

In [ ]:
print("--- Validación de RUTs ---")
for persona in personas:
    if persona.validarut():
        print(f"{persona.nombre} {persona.apellido} - RUT Válido")
        rut_validos += 1
    else:
        print(f"{persona.nombre} {persona.apellido} - RUT Inválido")
        rut_invalidos += 1
print(f"
Total RUTs Válidos: {rut_validos}")
print(f"Total RUTs Inválidos: {rut_invalidos}")
print("
")


### 3.2 Información General de Personas
Se muestra la representación de cada persona (alumno o profesor) cargada utilizando el método `__repr__` de cada clase.

In [ ]:
print("--- Información General de Personas ---")
for persona in personas:
    print(persona)
print("
")


### 3.3 Promedios y Situación Académica de Alumnos
Se calcula y muestra el promedio de notas y la situación académica (Aprobado/Reprobado) de cada alumno, y se contabilizan los aprobados y reprobados.

In [ ]:
print("--- Promedios y Situación Académica de Alumnos ---")
for alumno in alumnos:
    print(f"{alumno.nombre} {alumno.apellido}: Promedio = {alumno.promedio()} | Situación: {alumno.situacion_academica()}")
    if alumno.situacion_academica() == "Aprobado":
        alumnos_aprobados += 1
    else:
        alumnos_reprobados += 1
print(f"
Total Alumnos Aprobados: {alumnos_aprobados}")
print(f"Total Alumnos Reprobados: {alumnos_reprobados}")
print("
")


### 3.4 Alumno con Mejor Promedio
Se identifica al alumno con el promedio más alto entre todos los alumnos cargados.

In [ ]:
print("--- Alumno con Mejor Promedio ---")
for alumno in alumnos:
    if alumno.promedio() > mejor_promedio:
        mejor_promedio = alumno.promedio()
        mejor_alumno = alumno

if mejor_alumno:
    print(f"El mejor alumno es: {mejor_alumno.nombre} {mejor_alumno.apellido}")
    print(f"Asignatura: {mejor_alumno.asignatura}")
    print(f"Promedio: {mejor_promedio}")
else:
    print("No hay alumnos para determinar el mejor promedio.")
print("
")


### 3.5 Tipo de Contrato de Profesores
Se muestra el tipo de contrato (Jornada Completa/Parcial) de cada profesor, basado en sus horas de contrato.

In [ ]:
print("--- Tipo de Contrato de Profesores ---")
for profe in profesores:
    print(f"{profe.nombre} {profe.apellido} - {profe.tipo_contrato()}")
print("
")


### 3.6 Total de Horas Contratadas y Profesor con Más Horas
Se calcula el total de horas contratadas por todos los profesores y se identifica al profesor o profesores con la mayor cantidad de horas, manejando el caso de múltiples profesores con el mismo máximo de horas.

In [ ]:
print("--- Total de Horas Contratadas y Profesor con Más Horas ---")
for profesor in profesores:
    total_horas_contratadas += profesor.horas_contrato
    if profesor.horas_contrato > max_horas:
        max_horas = profesor.horas_contrato
        profesor_mas_horas = [f"{profesor.nombre} {profesor.apellido} ({profesor.asignatura}): {profesor.horas_contrato} horas"]
    elif profesor.horas_contrato == max_horas:
        profesor_mas_horas.append(f"{profesor.nombre} {profesor.apellido} ({profesor.asignatura}): {profesor.horas_contrato} horas")

print(f"Total de horas contratadas: {total_horas_contratadas}")
print(f"Profesor(es) con más horas: {", ".join(profesor_mas_horas)}")
print("
")


### 3.7 Alumnos que Necesitan Apoyo
Se listan los alumnos que necesitan apoyo académico, basándose en su asistencia (menor a 80%) o promedio (menor a 4.5).

In [ ]:
print("--- Alumnos que Necesitan Apoyo ---")
for alumno in alumnos:
    if alumno.necesita_apoyo():
        print(f"{alumno.nombre} {alumno.apellido} - Promedio: {alumno.promedio()} - Asistencia: {alumno.asistencia}%")
print("
")


### 3.8 Resumen por Asignatura
Se agrupan alumnos y profesores por asignatura para mostrar un resumen detallado, incluyendo el profesor asignado, la cantidad de alumnos y el promedio del curso.

In [ ]:
# Agrupar alumnos por asignatura
for alumno in alumnos:
    asignatura = alumno.asignatura
    if asignatura not in alumnos_por_asignatura:
        alumnos_por_asignatura[asignatura] = []
    alumnos_por_asignatura[asignatura].append(alumno)

# Asignar profesor a asignatura (asumiendo un profesor por asignatura para este resumen)
for profesor in profesores:
    profesores_por_asignatura[profesor.asignatura] = profesor

print("--- Resumen por Asignatura ---")
for asignatura, lista_alumnos in alumnos_por_asignatura.items():
    cantidad_alumnos = len(lista_alumnos)
    suma_promedios = sum(alumno.promedio() for alumno in lista_alumnos)
    promedio_curso = round(suma_promedios / cantidad_alumnos, 1) if cantidad_alumnos > 0 else 0

    profesor_asignado = profesores_por_asignatura.get(asignatura, None)
    profesor_info = f"{profesor_asignado.nombre} {profesor_asignado.apellido}" if profesor_asignado else "No asignado"

    print(f"
Asignatura: {asignatura}")
    print(f"Profesor: {profesor_info}")
    print(f"Cantidad de Alumnos: {cantidad_alumnos}")
    print(f"Promedio del Curso: {promedio_curso}")
    print("Alumnos:")
    for alumno in lista_alumnos:
        print(f"  - {alumno.nombre} {alumno.apellido} | Promedio: {alumno.promedio()} | {alumno.situacion_academica()}")
print("
")
